# 🎓 Módulo 4 — Desarrollo
**Sistema de Gestión de Recursos Humanos**

Este módulo cubre el desarrollo profesional de los colaboradores:
- **Capacitaciones** → registro y seguimiento de cursos/talleres
- **Evaluación de Desempeño** → puntuación bruta con penalización por ausencias → puntaje neto

> **Base de datos:** `rrhh_sistema` · **Puerto:** `1989` · **Motor:** MySQL


In [ ]:
# ── 1. Instalación automática de dependencias ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'mysql-connector-python', 'ipywidgets', 'pandas', '--quiet'], check=False)

import mysql.connector
from mysql.connector import Error
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import date

print('✅ Librerías cargadas correctamente.')


In [ ]:
# Conexion a la base de datos
import os
import mysql.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '1989')),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'rrhh_sistema'),
    'charset': 'utf8mb4',
}

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

try:
    conn = get_conn()
    conn.close()
    print(f'Conexion exitosa -> {DB_CONFIG["database"]} (Puerto {DB_CONFIG["port"]})')
except mysql.connector.Error as e:
    print(f'Error de conexion: {e}')



In [ ]:
# ── 3. Funciones auxiliares ───────────────────────────────────────────────

def get_empleados_activos() -> list:
    conn = get_conn(); cur = conn.cursor()
    cur.execute("""
        SELECT e.codigo_empresa,
               CONCAT(e.nombre, ' ', e.apellido, '  [', e.codigo_empresa, ']') AS label
        FROM   empleados e
        WHERE  e.estado = 'activo'
        ORDER  BY e.apellido, e.nombre
    """)
    rows = cur.fetchall(); cur.close(); conn.close()
    return rows

def registrar_capacitacion(codigo: str, nombre: str, fecha_inicio, fecha_fin) -> str:
    conn = get_conn(); cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO capacitaciones
                (codigo_empresa, nombre_capacitacion, fecha_inicio, fecha_fin)
            VALUES (%s, %s, %s, %s)
        """, (codigo, nombre, fecha_inicio, fecha_fin or None))
        conn.commit()
        return f'✅ Capacitación registrada: "{nombre}" para {codigo}'
    except Error as e:
        conn.rollback(); return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

def calcular_penalizacion(codigo: str) -> float:
    """Penaliza 2 puntos por ausencia en el mes actual (máximo 20 puntos)."""
    conn = get_conn(); cur = conn.cursor()
    cur.execute("""
        SELECT COUNT(*) FROM ausencias
        WHERE  codigo_empresa = %s
          AND  MONTH(fecha) = MONTH(CURDATE())
          AND  YEAR(fecha)  = YEAR(CURDATE())
    """, (codigo,))
    ausencias = cur.fetchone()[0]
    cur.close(); conn.close()
    return min(ausencias * 2.0, 20.0)

def registrar_evaluacion(codigo: str, fecha, pct_bruto: float) -> str:
    penalizacion = calcular_penalizacion(codigo)
    pct_neto     = max(pct_bruto - penalizacion, 0.0)
    conn = get_conn(); cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO evaluaciones_desempeno
                (codigo_empresa, fecha_evaluacion, pct_bruto, pct_neto)
            VALUES (%s, %s, %s, %s)
        """, (codigo, fecha, pct_bruto, pct_neto))
        # Actualizar promedio de desempeño en empleados
        cur.execute("""
            UPDATE empleados
            SET    pct_desempeno = (
                SELECT ROUND(AVG(pct_neto), 2)
                FROM   evaluaciones_desempeno
                WHERE  codigo_empresa = %s
            )
            WHERE  codigo_empresa = %s
        """, (codigo, codigo))
        conn.commit()
        return (f'✅ Evaluación guardada — Bruto: {pct_bruto:.1f}  '
                f'Penalización: -{penalizacion:.1f}  Neto: {pct_neto:.1f}')
    except Error as e:
        conn.rollback(); return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

print('✅ Funciones auxiliares cargadas.')


In [ ]:
# ── 4. PANEL PRINCIPAL CON PESTAÑAS ──────────────────────────────────────

STYLE  = {'description_width': '180px'}
LAYOUT = widgets.Layout(width='430px')
BTN_LY = widgets.Layout(width='220px', margin='8px 4px 0 0')
OUT_LY = widgets.Layout(margin='8px 0 0 0', border='1px solid #e0e0e0',
                         padding='10px', border_radius='6px', min_height='40px')

emp_raw  = get_empleados_activos()
emp_opts = {label: codigo for codigo, label in emp_raw}

def emp_dropdown(desc='Empleado *'):
    return widgets.Dropdown(description=desc, options=list(emp_opts.keys()),
                             style=STYLE, layout=LAYOUT)

# ══════════════════════════════════════════════════════════════════════════
# TAB 1 — CAPACITACIONES
# ══════════════════════════════════════════════════════════════════════════
w_c_emp    = emp_dropdown('Empleado *')
w_c_nombre = widgets.Text(description='Nombre capacitación *',
                           placeholder='Ej. Excel Avanzado, Liderazgo...',
                           style=STYLE, layout=LAYOUT)
w_c_inicio = widgets.DatePicker(description='Fecha inicio *', value=date.today(),
                                 style=STYLE, layout=LAYOUT)
w_c_fin    = widgets.DatePicker(description='Fecha fin (opcional)',
                                 style=STYLE, layout=LAYOUT)
btn_c_guardar = widgets.Button(description='💾 Registrar Capacitación',
                                button_style='success', layout=BTN_LY)
out_c = widgets.Output(layout=OUT_LY)

btn_c_ver = widgets.Button(description='📋 Ver capacitaciones activas',
                            button_style='info', layout=BTN_LY)
out_c_ver = widgets.Output(layout=OUT_LY)

def on_guardar_cap(b):
    with out_c:
        clear_output()
        if not emp_opts: print('⚠️ No hay empleados activos.'); return
        if not w_c_nombre.value.strip():
            print('⚠️ El nombre de la capacitación es requerido.'); return
        codigo = emp_opts[w_c_emp.value]
        print(registrar_capacitacion(codigo, w_c_nombre.value.strip(),
                                     w_c_inicio.value, w_c_fin.value))

def on_ver_cap(b):
    with out_c_ver:
        clear_output()
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT CONCAT(e.nombre,' ',e.apellido) AS empleado,
                       c.nombre_capacitacion, c.fecha_inicio, c.fecha_fin
                FROM   capacitaciones c
                JOIN   empleados e ON e.codigo_empresa = c.codigo_empresa
                WHERE  c.fecha_fin IS NULL OR c.fecha_fin >= CURDATE()
                ORDER  BY c.fecha_inicio DESC
            """, conn); conn.close()
            if df.empty: print('No hay capacitaciones activas.')
            else:
                display(HTML("<b>Capacitaciones activas / en curso</b>"))
                display(df.style.hide(axis='index'))
        except Exception as ex:
            print(f'❌ Error: {ex}')

btn_c_guardar.on_click(on_guardar_cap)
btn_c_ver.on_click(on_ver_cap)

tab1 = widgets.VBox([
    widgets.HTML("<h3 style='color:#1a73e8'>📚 Registrar Capacitación</h3>"),
    w_c_emp, w_c_nombre, w_c_inicio, w_c_fin,
    widgets.HBox([btn_c_guardar]), out_c,
    widgets.HTML("<hr>"),
    widgets.HBox([btn_c_ver]), out_c_ver
], layout=widgets.Layout(padding='16px'))

# ══════════════════════════════════════════════════════════════════════════
# TAB 2 — EVALUACIÓN DE DESEMPEÑO
# ══════════════════════════════════════════════════════════════════════════
w_e_emp   = emp_dropdown('Empleado *')
w_e_fecha = widgets.DatePicker(description='Fecha evaluación *', value=date.today(),
                                style=STYLE, layout=LAYOUT)
w_e_bruto = widgets.BoundedFloatText(description='Puntuación bruta (0-100) *',
                                      value=80.0, min=0.0, max=100.0, step=0.5,
                                      style=STYLE, layout=LAYOUT)
out_e_info = widgets.Output(layout=OUT_LY)
btn_e_preview = widgets.Button(description='🔍 Ver penalización',
                                button_style='warning',
                                layout=widgets.Layout(width='200px', margin='4px 4px 0 0'))
btn_e_guardar = widgets.Button(description='💾 Guardar Evaluación',
                                button_style='success', layout=BTN_LY)
out_e = widgets.Output(layout=OUT_LY)

btn_e_ver = widgets.Button(description='📊 Ver historial desempeño',
                            button_style='info', layout=BTN_LY)
out_e_ver = widgets.Output(layout=OUT_LY)

def on_preview_eval(b):
    with out_e_info:
        clear_output()
        if not emp_opts: print('⚠️ No hay empleados activos.'); return
        codigo = emp_opts[w_e_emp.value]
        pen    = calcular_penalizacion(codigo)
        neto   = max(w_e_bruto.value - pen, 0.0)
        display(HTML(f"""
        <div style='background:#fff8e1;border:1px solid #ffc107;border-radius:6px;padding:10px'>
            <b>Resumen de penalización (mes actual)</b><br>
            Ausencias registradas este mes → penalización: <b>-{pen:.1f} pts</b><br>
            Puntuación bruta: <b>{w_e_bruto.value:.1f}</b> &minus; {pen:.1f} = Neto: <b>{neto:.1f}</b>
        </div>"""))

def on_guardar_eval(b):
    with out_e:
        clear_output()
        if not emp_opts: print('⚠️ No hay empleados activos.'); return
        codigo = emp_opts[w_e_emp.value]
        print(registrar_evaluacion(codigo, w_e_fecha.value, w_e_bruto.value))

def on_ver_eval(b):
    with out_e_ver:
        clear_output()
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT CONCAT(e.nombre,' ',e.apellido) AS empleado,
                       ev.fecha_evaluacion, ev.pct_bruto, ev.pct_neto,
                       ROUND(e.pct_desempeno, 2) AS promedio_neto
                FROM   evaluaciones_desempeno ev
                JOIN   empleados e ON e.codigo_empresa = ev.codigo_empresa
                ORDER  BY ev.fecha_evaluacion DESC
                LIMIT  50
            """, conn); conn.close()
            if df.empty: print('No hay evaluaciones registradas.')
            else:
                display(HTML("<b>Últimas 50 evaluaciones</b>"))
                display(df.style.hide(axis='index'))
        except Exception as ex:
            print(f'❌ Error: {ex}')

btn_e_preview.on_click(on_preview_eval)
btn_e_guardar.on_click(on_guardar_eval)
btn_e_ver.on_click(on_ver_eval)

tab2 = widgets.VBox([
    widgets.HTML("<h3 style='color:#6a1b9a'>📊 Evaluación de Desempeño</h3>"),
    w_e_emp, w_e_fecha, w_e_bruto,
    widgets.HTML("<small style='color:#666'>💡 La penalización es -2 pts por ausencia en el mes (máx -20 pts)</small>"),
    widgets.HBox([btn_e_preview, btn_e_guardar]), out_e_info, out_e,
    widgets.HTML("<hr>"),
    widgets.HBox([btn_e_ver]), out_e_ver
], layout=widgets.Layout(padding='16px'))

# ══════════════════════════════════════════════════════════════════════════
# PANEL FINAL
# ══════════════════════════════════════════════════════════════════════════
panel = widgets.Tab(children=[tab1, tab2])
panel.set_title(0, '📚 Capacitaciones')
panel.set_title(1, '📊 Evaluaciones')

display(panel)
